Price Prediction Learning Curve Experiment using LSTM and CNN

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer

In [ ]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HG-TOKEN']
login(hf_token, add_to_git_credential=True)

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
# Prepare text + labels

documents = [item.summary for item in train]
y = np.array([float(item.price) for item in train])

# Tokenizer (BERT tokenizer)

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 64

def encode_texts(texts):
    encodings = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )
    return encodings["input_ids"]

X = encode_texts(documents)

# Convert labels to tensor

y_tensor = torch.FloatTensor(y).unsqueeze(1)

# Train/validation split

X_train, X_val, y_train, y_val = train_test_split(X, y_tensor, test_size=0.01, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

VOCAB_SIZE = tokenizer.vocab_size
EMBED_DIM = 128
HIDDEN_DIM = 128

In [ ]:
# -------------------------
# LSTM MODEL
# -------------------------

class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM)

        self.lstm = nn.LSTM(
            input_size=EMBED_DIM,
            hidden_size=HIDDEN_DIM,
            batch_first=True
        )

        self.fc = nn.Linear(HIDDEN_DIM, 1)

    def forward(self, x):

        x = self.embedding(x)

        _, (hidden, _) = self.lstm(x)

        output = self.fc(hidden[-1])

        return output


In [ ]:
# -------------------------
# CNN MODEL
# -------------------------

class CNNModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM)

        self.conv1 = nn.Conv1d(EMBED_DIM, 128, kernel_size=3)
        self.conv2 = nn.Conv1d(EMBED_DIM, 128, kernel_size=4)
        self.conv3 = nn.Conv1d(EMBED_DIM, 128, kernel_size=5)

        self.pool = nn.AdaptiveMaxPool1d(1)

        self.fc = nn.Linear(128 * 3, 1)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.embedding(x)

        x = x.permute(0, 2, 1)

        c1 = self.pool(self.relu(self.conv1(x))).squeeze(2)
        c2 = self.pool(self.relu(self.conv2(x))).squeeze(2)
        c3 = self.pool(self.relu(self.conv3(x))).squeeze(2)

        x = torch.cat((c1, c2, c3), dim=1)

        output = self.fc(x)

        return output


In [ ]:
# Choose model

#model = LSTMModel()
model = CNNModel()

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Training setup

train_losses = []
val_losses = []

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 3 # gave better results for CNN and EPOCHS = 5 for LSTM

for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0

    for batch_X, batch_y in tqdm(train_loader):

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)

    model.eval()
    with torch.no_grad():

        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val).item()
        val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {epoch_loss:.3f} - Val Loss: {val_loss:.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()

plt.show()

In [ ]:
# Prediction function

def neural_network(item):

    model.eval()

    with torch.no_grad():

        vector = encode_texts([item.summary])

        result = model(vector)[0].item()

    return max(0, result)


evaluate(neural_network, test)

In [ ]:
true_prices = []
pred_prices = []

model.eval()

with torch.no_grad():

    for item in test:

        pred = neural_network(item)

        true_prices.append(float(item.price))
        pred_prices.append(pred)

plt.figure(figsize=(6,6))

plt.scatter(true_prices, pred_prices, alpha=0.3)

plt.plot([0, max(true_prices)], [0, max(true_prices)], 'r--')

plt.xlabel("True Price")
plt.ylabel("Predicted Price")

plt.title("Predicted vs True Prices")

plt.show()